# XFeat-SuperPoint Hybrid — MegaDepth Raw Training (Robust Colab)

This notebook performs a hardened end-to-end setup for training on **MegaDepth raw scene folders**.
It validates runtime, installs dependencies with retries, clones required upstream repos, downloads weights, validates dataset layout, and launches training with checkpoint resume support.

In [ ]:
#@title 1) Runtime preflight (GPU + Python)
import os, sys, subprocess, textwrap

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU runtime detected. In Colab: Runtime -> Change runtime type -> GPU.')

print(result.stdout.splitlines()[0])
print('Python:', sys.version.split()[0])
print('Executable:', sys.executable)


In [ ]:
#@title 2) Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
#@title 3) Paths and training knobs
from pathlib import Path

# ===== User-configurable =====
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/hybrid_feature_matching'
RAW_ROOT_DRIVE     = f'{DRIVE_PROJECT_ROOT}/data/megadepth_raw'   # contains scene dirs like 0001, 0002, ...
RAW_TAR_PATH       = ''  # optional tar on Drive, e.g. '/content/drive/MyDrive/.../megadepth_raw.tar'

REPO_URL           = 'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git'
REPO_DIR           = '/content/XFeat-SuperPoint-Hybrid-Model'
XFEAT_REPO_URL     = 'https://github.com/verlab/accelerated_features.git'
XFEAT_DIR          = '/content/accelerated_features'
SUPERPOINT_REPO_URL= 'https://github.com/rpautrat/SuperPoint.git'
SUPERPOINT_DIR     = '/content/SuperPoint'

DRIVE_CKPT_DIR     = f'{DRIVE_PROJECT_ROOT}/checkpoints/megadepth_raw'
DRIVE_LOG_DIR      = f'{DRIVE_PROJECT_ROOT}/runs/megadepth_raw'

# Training hyperparameters
BATCH_SIZE         = 4
MAX_EPOCHS         = 50
LR                 = 1e-4
NUM_WORKERS        = 2
IMAGE_H, IMAGE_W   = 480, 640  # MUST be divisible by 8
VAL_SPLIT_RATIO    = 0.2
MAX_PAIRS_PER_SCENE = 200
MIN_OVERLAP        = 0.15
MAX_OVERLAP        = 0.95
MIN_KEYPOINT_SCORE = 0.01

# Resume checkpoint: leave empty to auto-pick best.pth / latest epoch_*.pth
RESUME_CKPT        = ''
NO_VERIFY_DATASET_PAIRS = False
VALIDATION_SCAN_LIMIT    = 200
OVERRIDE_CFG_PATH        = '/content/megadepth_raw_override.yaml'

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_LOG_DIR).mkdir(parents=True, exist_ok=True)

print('REPO_DIR       :', REPO_DIR)
print('RAW_ROOT_DRIVE :', RAW_ROOT_DRIVE)
print('DRIVE_CKPT_DIR :', DRIVE_CKPT_DIR)
print('DRIVE_LOG_DIR  :', DRIVE_LOG_DIR)


In [ ]:
#@title 4) Clone/update repos + install dependencies (with retries)
import os, sys, subprocess, time
from pathlib import Path


def run_with_retry(cmd, cwd=None, retries=1, sleep_s=5):
    last = None
    for attempt in range(1, retries + 1):
        try:
            print('+', ' '.join(cmd))
            subprocess.check_call(cmd, cwd=cwd)
            return
        except subprocess.CalledProcessError as e:
            last = e
            if attempt < retries:
                print(f'Command failed (attempt {attempt}/{retries}), retrying in {sleep_s}s...')
                time.sleep(sleep_s)
    raise last


def ensure_repo(path, url):
    p = Path(path)
    if p.exists() and (p / '.git').exists():
        run_with_retry(['git', '-C', str(p), 'fetch', '--all'], retries=2)
        run_with_retry(['git', '-C', str(p), 'pull', '--ff-only'], retries=2)
    elif p.exists():
        raise RuntimeError(f'Path exists but is not a git repo: {p}')
    else:
        run_with_retry(['git', 'clone', url, str(p)], retries=2)


ensure_repo(REPO_DIR, REPO_URL)
ensure_repo(XFEAT_DIR, XFEAT_REPO_URL)
ensure_repo(SUPERPOINT_DIR, SUPERPOINT_REPO_URL)

run_with_retry([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], retries=3)
run_with_retry([sys.executable, '-m', 'pip', 'install', '-r', f'{REPO_DIR}/requirements.txt'], retries=3)
run_with_retry([sys.executable, '-m', 'pip', 'install', 'h5py', 'kornia', 'matplotlib', 'lightglue'], retries=3)

# Make upstream repos importable in this runtime and subprocesses.
os.environ['PYTHONPATH'] = ':'.join([
    REPO_DIR,
    XFEAT_DIR,
    SUPERPOINT_DIR,
    os.environ.get('PYTHONPATH', ''),
]).strip(':')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if XFEAT_DIR not in sys.path:
    sys.path.insert(0, XFEAT_DIR)
if SUPERPOINT_DIR not in sys.path:
    sys.path.insert(0, SUPERPOINT_DIR)

print('Dependency setup complete.')
print('PYTHONPATH=', os.environ['PYTHONPATH'])


In [ ]:
#@title 5) Download pretrained weights (idempotent)
import urllib.request
from pathlib import Path

weights_dir = Path(REPO_DIR) / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)

assets = [
    ('https://github.com/verlab/accelerated_features/releases/download/v1.0/xfeat.pt', weights_dir / 'xfeat.pth'),
    ('https://github.com/magicleap/SuperPointPretrainedNetwork/raw/master/superpoint_v1.pth', weights_dir / 'superpoint_v1.pth'),
]

for url, dst in assets:
    if dst.exists() and dst.stat().st_size > 0:
        print('Exists:', dst)
        continue
    print('Downloading', url, '->', dst)
    urllib.request.urlretrieve(url, str(dst))
    if dst.stat().st_size == 0:
        raise RuntimeError(f'Downloaded empty file: {dst}')

print('Weights ready in', weights_dir)


In [ ]:
#@title 6) Optional: extract RAW_TAR_PATH to /content/megadepth_raw
import tarfile
from pathlib import Path

if RAW_TAR_PATH:
    tar_path = Path(RAW_TAR_PATH)
    if not tar_path.exists():
        raise FileNotFoundError(f'RAW_TAR_PATH not found: {tar_path}')

    extract_root = Path('/content/megadepth_raw')
    extraction_flag = extract_root / '.extracted'
    if extraction_flag.exists():
        print('Already extracted:', extract_root)
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        print('Extracting', tar_path, '->', extract_root)
        with tarfile.open(tar_path, 'r:*') as tf:
            tf.extractall(extract_root)
        if not any(extract_root.glob('**/dense0/imgs/*')):
            raise RuntimeError('Extraction completed but no images found under **/dense0/imgs')
        extraction_flag.write_text('ok')
    RAW_ROOT = str(extract_root)
else:
    RAW_ROOT = RAW_ROOT_DRIVE

print('RAW_ROOT:', RAW_ROOT)


In [ ]:
#@title 7) Validate MegaDepth raw layout
from pathlib import Path

raw = Path(RAW_ROOT)
if not raw.exists():
    raise FileNotFoundError(f'RAW_ROOT does not exist: {raw}')

scene_dirs = sorted([p for p in raw.iterdir() if p.is_dir()])
img_dirs = list(raw.glob('**/dense0/imgs'))
depth_dirs = list(raw.glob('**/dense0/depths'))

num_imgs = 0
for d in img_dirs[:VALIDATION_SCAN_LIMIT]:
    num_imgs += len(list(d.glob('*.jpg'))) + len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.png')))

num_h5 = 0
for d in depth_dirs[:VALIDATION_SCAN_LIMIT]:
    num_h5 += len(list(d.glob('*.h5')))

print('scene dirs:', len(scene_dirs))
print('img dirs  :', len(img_dirs))
print('depth dirs:', len(depth_dirs))
print('sample image count (partial scan):', num_imgs)
print('sample depth count (partial scan):', num_h5)

if len(img_dirs) == 0 or num_imgs == 0:
    raise RuntimeError('No training images found under **/dense0/imgs. Check RAW_ROOT.')


In [ ]:
#@title 8) Training preflight import checks
import os
import subprocess
import sys

env = os.environ.copy()
env['PYTHONPATH'] = ':'.join([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR, env.get('PYTHONPATH', '')]).strip(':')

check_script = (
    "import importlib, torch; "
    "print('torch', torch.__version__); "
    "a=importlib.import_module('modules.xfeat'); assert hasattr(a,'XFeat'); print('XFeat import ok'); "
    "sp=None; "
    "c=[('models.superpoint_core','SuperPointCore'),('superpoint.superpoint','SuperPoint'),('superpoint_pytorch','SuperPoint'),('superpoint','SuperPoint')]; "
    "\nfor m,n in c:\n"
    "    try:\n"
    "        mod=importlib.import_module(m);\n"
    "        if hasattr(mod,n): sp=(m,n); break\n"
    "    except Exception:\n"
    "        pass\n"
    "\nassert sp is not None, 'No compatible SuperPoint import path found'; print('SuperPoint import ok via', sp); "
    "import train; print('train import ok')"
)

cmd = [sys.executable, '-c', check_script]
print('+', ' '.join(cmd))
subprocess.check_call(cmd, cwd=REPO_DIR, env=env)
print('Preflight import checks passed.')


In [ ]:
#@title 9) Select resume checkpoint (auto-pick if empty)
from pathlib import Path

resume = RESUME_CKPT.strip()
if resume and not Path(resume).exists():
    raise FileNotFoundError(f'RESUME_CKPT does not exist: {resume}')

if not resume:
    ckpt_dir = Path(DRIVE_CKPT_DIR)
    best = ckpt_dir / 'best.pth'
    if best.exists():
        resume = str(best)
    else:
        epochs = sorted(ckpt_dir.glob('epoch_*.pth'))
        if epochs:
            resume = str(epochs[-1])

print('Using resume checkpoint:', resume if resume else '(none)')


In [ ]:
#@title 10) Launch MegaDepth Raw training
import os, subprocess, sys
from pathlib import Path

env = os.environ.copy()
env['PYTHONPATH'] = ':'.join([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR, env.get('PYTHONPATH', '')]).strip(':')

# train.py does not expose every config key as CLI args; write explicit overrides.
override_cfg_path = Path(OVERRIDE_CFG_PATH)
override_cfg = f"""
mode: megadepth_raw
data_root: {RAW_ROOT}
checkpoint_dir: {DRIVE_CKPT_DIR}
log_dir: {DRIVE_LOG_DIR}
batch_size: {BATCH_SIZE}
max_epochs: {MAX_EPOCHS}
lr: {LR}
num_workers: {NUM_WORKERS}
image_height: {IMAGE_H}
image_width: {IMAGE_W}
megadepth_val_split_ratio: {VAL_SPLIT_RATIO}
max_pairs_per_scene: {MAX_PAIRS_PER_SCENE}
min_overlap: {MIN_OVERLAP}
max_overlap: {MAX_OVERLAP}
min_keypoint_score: {MIN_KEYPOINT_SCORE}
""".strip() + "\n"
override_cfg_path.write_text(override_cfg)
print('Wrote config override to', override_cfg_path)
print(override_cfg)

cmd = [
    sys.executable, f'{REPO_DIR}/train.py',
    '--config', str(override_cfg_path),
]

if NO_VERIFY_DATASET_PAIRS:
    cmd.append('--no_verify_dataset_pairs')
if resume:
    cmd += ['--resume', resume]

print('Running command:\n', ' '.join(cmd))
subprocess.check_call(cmd, cwd=REPO_DIR, env=env)


In [ ]:
#@title 11) List saved checkpoints
from pathlib import Path

ckpts = sorted(Path(DRIVE_CKPT_DIR).glob('*.pth'))
print('Checkpoint dir:', DRIVE_CKPT_DIR)
print('Found', len(ckpts), 'checkpoint files')
for p in ckpts[-30:]:
    print('-', p.name)
